.py: 
- read_csv('seismic-bumps.csv') yang tidak memakai DATA_PATH
- ada baris pip install telanjang, 
- !pip ala Colab
- header docstring ganda, 
- blok loop sisa yang kosong, 

"""
Seismic Bumps — Preprocessing-Leak-Free Pipeline v3 (SMOTETomek + tuning of the CV-best model)
=============================================================
Flow:
  1. Load + deterministic cleaning only (dedup, constant cols). NO outlier removal, NO PCA.
  2. Stratified 80/20 split (or temporal split via SPLIT_MODE). Test set sealed.
  3. Repeated Stratified K-Fold CV on TRAIN only. Every fit (imputer, encoder,
     scaler, SMOTETomek, classifier) lives inside ONE imblearn Pipeline, so it is
     refit on each training fold. Validation fold = transform/predict only.
  4. Model selection = CV PR-AUC on train, ALL models with DEFAULT hyperparameters
     (fair comparison). Test never used to choose.
  5. [NEW] Hyperparameter tuning of the CV-selected model ONLY (RandomizedSearchCV on
     train, inner CV, scoring = average_precision). Tuned vs default is then re-checked
     on FRESH CV splits; tuned params are adopted only if they win there.
  6. OOF probabilities (fresh splits, from the adopted config) -> threshold grid search
     (0.01..0.99) on the minority class.
  7. Final fit on full train -> evaluate on test ONCE (+ bootstrap CI).
  8. XAI: permutation importance + SHAP (post-hoc only).
  9. Save the WHOLE pipeline + threshold in one artifact.

Scope of the "leak-free" claim: imputation, encoding, scaling, resampling, tuning and model selection never
see validation/test data (no PREPROCESSING / RESAMPLING leakage). Random stratified CV on shifts that are
consecutive in time can still be optimistic because of temporal autocorrelation (see E4).

Paper-guide experiments (Panduan_Paper_Seismic_Bump.md / extra_experiments.py), inserted at the matching step.
Select with EXTRA_RUN (e.g. EXTRA_RUN=e1,e2,e6 | all | none). Outputs -> results_extra/
  4B. E2  paired statistical tests between models (Nadeau-Bengio + Holm)        -> tests H3
  4C. E1  leaky vs leak-free protocol (scaling leak, global-SMOTENC leak)        -> tests H1
  7B. E4  temporal split vs random split (needs original row order)             -> limitation
  7C. E5  cost analysis of the threshold (FN:FP ratios)                          -> operational implication
  8C. E3  feature-importance consistency across folds and models                 -> tests H2
  8D. E6  feature-group ablation: pulse vs energy vs bumps vs qualitative        -> tests H2

Imbalance handling per model (never stacked on each other):
  - class_weight models      : weights computed inside fit() on the training fold only
  - SMOTETomek models        : SMOTENC (categorical-aware oversampling) + Tomek-link cleaning,
                               inside the fold, after encode+scale
  - SPE / BalanceCascade / SMOTEBagging : balance internally (no external resampler)
  - Stacking                 : class_weight ONLY (SMOTE before an inner CV leaks
                               synthetic neighbours across the inner folds)
"""

# Requirements:  pip install scikit-learn imbalanced-learn imbalanced-ensemble lightgbm scipy pandas matplotlib joblib shap
#                (shap is optional; deep-forest is optional and not part of the final model set)
# Usage:         python seismic7_final.py          (reads seismic-bumps.csv from the working directory, or set DATA_PATH)


# BUILDING BLOCKS:
   Make Preprocessor() : Imputer + ordinal encoder (cat) and imputer + StandardScaler (numeric ONLY).
    Output column order = [categorical..., numeric...] so SMOTENC indices are 0..n_cat-1.
    Lives INSIDE the pipeline => fit on the training fold only, transform-only on val/test.

   Make Sampler() : SMOTETomek = oversample minority (SMOTE) + clean overlapping boundary pairs (Tomek links).
    Plain SMOTE would interpolate the ordinal-encoded categorical codes into meaningless fractions,
    so SMOTENC is plugged in as the oversampler inside SMOTETomek (categorical-aware).
    Only active during fit() — never touches val/test.

   minority_report() : for class (1 = Hazardous) metric suite. No accuracy / F1-macro (both hide the minority).

   find_optimal_threshold() : Grid search t in [0.01, 0.99] maximising the minority-class F-beta on OOF predictions.

   cv_oof() : Repeated stratified CV. Each fold: fresh pipeline fit on the train fold ONLY (imputer/encoder/scaler/     SMOTETomek/classifier), predict_proba on the untouched val fold.

   bootstrap_ci() : Stratified bootstrap CI — with ~34 test positives a point estimate alone is misleading.

   run_extra(): Run one extra experiment if selected by EXTRA_RUN. A failure is reported but never kills the main pipeline.

   make_preprocessor_subset() : E6 only: same preprocessing as make_preprocessor(), restricted to the feature subset cols
   
   nadeau_bengio_t() : Corrected resampled t-test (Nadeau & Bengio, 2003) untuk repeated k-fold CV, and d = score diff per fold (model A - model B). correction: (1/k + n_test/n_train).

   holm() : The Holm-Bonferroni correction for multiple comparisons.

   e1_leaky_vs_leakfree(): 
      Three protocols on the SAME DATA and the same fold:
      A. leak-free       : all preprocessing + (SMOTENC) within the fold (pipeline)
      B. leaky-scaling   : imputer/encoder/scaler fitted on ALL data before CV
      C. leaky-SMOTE     : SMOTENC applied to ALL data before CV (classic leakage as described in the literature)
            C-all  : evaluated on all validation rows (including synthetic samples)
            C-orig: evaluated ONLY on the original rows (separating the leakage effect from the effect of
                     changes in class prevalence; PR-AUC depends on prevalence)
      A2. in-fold SMOTENC: SMOTENC (without Tomek) IN the fold; the classifier is the SAME as in C.
            Net comparison: A2 - A = pure oversampling effect; C-orig - A2 = pure leakage effect
            (C-orig - A combines both).
      Model: RandomForest class_weight (A, A2, B, C) + LogReg class_weight (A, B; scale-sensitive).
      Expectation: B ~ 0 (small-scale leakage), C-all and C-orig are much higher than A.


   e4_temporal_split(model_names=("RandomForest (class_weight)", "LogReg (class_weight)")):
    Train on the first 80% of rows, test on the last 20% of rows. Compare with a random split.
    WARNING: If the CSV file has already been randomized, these results are meaningless. Check the order first.
    The threshold is selected from OOF in the training set (StratifiedKFold) -> remains leak-free from the test set.
    

   e5_cost_analysis(model_name="RandomForest (class_weight)", cost_ratios=(1, 2, 5, 10, 20, 50)): Relative cost: FP = 1 (inspection/brief stop), FN = ratio (undetected bump).For each ratio, select the threshold that minimizes the total cost on the OOF (train), then calculate the ACTUAL cost on the test set. Compare with “never alarm” and “always alarm."

   e3_feature_consistency(model_names=("RandomForest (class_weight)", "LightGBM (class_weight)")):Permutation importance (drop PR-AUC) for EACH validation fold of the 5-fold x R repeat.
    Each validation fold contains ~34 positives -> much more stable than a single test set.
    Output: average importance, standard deviation across folds, average rank, frequency of top-3 placements.
    
   e6_group_ablation(base="RandomForest (class_weight)"): Train the SAME model with different feature subsets on the SAME fold, then test it in pairs against the model with all features. Two directions: “only group X” and “all except X.” If the hypothesis is true: pulse-only >> energy-only, and “all except pulse” shows a significant decrease.
    


In [1]:
import os
import json
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, fbeta_score, matthews_corrcoef, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV, RepeatedStratifiedKFold, StratifiedKFold, train_test_split,
)
from scipy.stats import loguniform, randint, uniform
from scipy import stats                              # paired tests, Spearman
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline as ImbPipeline
from imbens.ensemble import (
    BalanceCascadeClassifier, SelfPacedEnsembleClassifier, SMOTEBaggingClassifier,
)
from lightgbm import LGBMClassifier

try:                                            # Deep Forest is optional (heavy / pip: deep-forest)
    from deepforest import CascadeForestClassifier
    HAS_DEEP_FOREST = True
except Exception:
    HAS_DEEP_FOREST = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Set Config
DATA_PATH     = os.getenv("DATA_PATH", "seismic-bumps.csv")
TARGET        = "class"
RANDOM_STATE  = 42
TEST_SIZE     = 0.20
SPLIT_MODE    = os.getenv("SPLIT_MODE", "stratified")   # "stratified" | "temporal" (last 20% of rows = test; needs original row order)
N_SPLITS      = int(os.getenv("N_SPLITS", 10))           # ~170 positives -> 10 Folds
N_REPEATS     = int(os.getenv("N_REPEATS", 3))          # repeated CV = more stable estimate
THRESH_GRID   = np.round(np.arange(0.01, 1.00, 0.01), 2)
THRESH_BETA   = 1.0                                     # 1.0 = F1 (as requested); 2.0 = recall-weighted F2 (safety-critical option)
N_BOOT        = int(os.getenv("N_BOOT", 1000))          # bootstrap resamples for test CI
TOMEK_STRATEGY = "all"                                  # "all" = classic SMOTETomek (remove both ends of a Tomek link); "majority" = never delete minority samples
TUNE_MODEL    = os.getenv("TUNE_MODEL", "auto")         # "auto" = CV-best model (if it has a search space) | exact model name | "none"
TUNE_N_ITER   = int(os.getenv("TUNE_N_ITER", 25))       # random-search candidates (stacking is slow: each candidate = 5 folds x 2 repeats)
TUNE_REPEATS  = int(os.getenv("TUNE_REPEATS", 2))
TUNE_N_JOBS   = int(os.getenv("TUNE_N_JOBS", 1))        # RF inside stacking already uses all cores -> keep 1 to avoid oversubscription
REPORT_ALL_ON_TEST = False                              # False = strict: ONLY the CV-selected model touches the test set
OUT_DIR       = "results"
os.makedirs(OUT_DIR, exist_ok=True)

QUICK         = os.getenv("QUICK", "0") == "1"           # QUICK=1 -> smoke test only (fewer repeats/folds); NOT for paper numbers
EXTRA_RUN     = os.getenv("EXTRA_RUN", "all").lower()    # "all" | "none" | comma list, e.g. "e1,e2,e6"
EXTRA_DIR     = "results_extra"
os.makedirs(EXTRA_DIR, exist_ok=True)

# a < b < c < d is a real order in the UCI doc (lack of hazard -> danger state)
ORDINAL_ORDER = {
    "seismic": ["a", "b", "c", "d"],
    "seismoacoustic": ["a", "b", "c", "d"],
    "hazard": ["a", "b", "c", "d"],
    "shift": ["N", "W"],
}
CONSTANT_COLS = ["nbumps6", "nbumps7", "nbumps89"]      # zero-variance / duplicate columns in this dataset

In [ ]:
t0_all = time.perf_counter()

# LOAD + DETERMINISTIC CLEANING (row-wise / schema-only — no statistics learned)
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape}  | missing cells: {int(df.isnull().sum().sum())}")
print(f"Raw class counts BEFORE de-duplication: 0={int((df[TARGET] == 0).sum())}  1={int((df[TARGET] == 1).sum())}")   #report raw vs de-duplicated N

n0 = len(df)
df = df.drop_duplicates()                               # BEFORE split: identical rows can't land in both train & test
df = df.loc[:, ~df.columns.duplicated()]
df = df.drop(columns=[c for c in CONSTANT_COLS if c in df.columns])
print(f"Dropped {n0 - len(df)} duplicate rows -> {df.shape}")
# NOTE: no boxplot/IQR outlier removal. High-energy events ARE the signal here, and
# IQR bounds computed on all data would leak test statistics. Trees are outlier-robust;
# the scaler is fit in-fold only.

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)
print(f"Class balance: 0={int((y == 0).sum())}  1={int((y == 1).sum())}  (prevalence={y.mean():.4f} "
      f"= PR-AUC of a random classifier)")

Loaded (2584, 19)  | missing cells: 0
Raw class counts BEFORE de-duplication: 0=2414  1=170
Dropped 6 duplicate rows -> (2578, 16)
Class balance: 0=2408  1=170  (prevalence=0.0659 = PR-AUC of a random classifier)


In [ ]:
# SPLIT — test set sealed until the very end
if SPLIT_MODE == "temporal":
    cut = int(len(X) * (1 - TEST_SIZE))
    X_train, X_test = X.iloc[:cut].reset_index(drop=True), X.iloc[cut:].reset_index(drop=True)
    y_train, y_test = y.iloc[:cut].reset_index(drop=True), y.iloc[cut:].reset_index(drop=True)
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    X_train, X_test = X_train.reset_index(drop=True), X_test.reset_index(drop=True)
    y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

y_train_np, y_test_np = y_train.to_numpy(), y_test.to_numpy()
print(f"Split [{SPLIT_MODE}] train={X_train.shape} (pos={int(y_train.sum())}) | "
      f"test={X_test.shape} (pos={int(y_test.sum())})  <- few positives => report CI, expect noise")

cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]
n_cat = len(cat_cols)
# category ORDER is domain knowledge (hard-coded); unseen columns fall back to train categories
cat_categories = [ORDINAL_ORDER.get(c, sorted(X_train[c].dropna().unique().tolist())) for c in cat_cols]
print(f"Categorical: {cat_cols}\nNumeric    : {num_cols}")

Split [stratified] train=(2062, 15) (pos=136) | test=(516, 15) (pos=34)  <- few positives => report CI, expect noise
Categorical: ['seismic', 'seismoacoustic', 'shift', 'hazard']
Numeric    : ['genergy', 'gpuls', 'gdenergy', 'gdpuls', 'nbumps', 'nbumps2', 'nbumps3', 'nbumps4', 'nbumps5', 'energy', 'maxenergy']


In [ ]:
# BUILDING BLOCKS
def make_preprocessor():
    cat_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OrdinalEncoder(categories=cat_categories,
                               handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    num_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ])
    return ColumnTransformer([("cat", cat_pipe, cat_cols), ("num", num_pipe, num_cols)],
                             verbose_feature_names_out=False)


def make_sampler():
    smote = (SMOTENC(categorical_features=list(range(n_cat)), random_state=RANDOM_STATE)
             if n_cat > 0 else SMOTE(random_state=RANDOM_STATE))
    return SMOTETomek(smote=smote, tomek=TomekLinks(sampling_strategy=TOMEK_STRATEGY),
                      random_state=RANDOM_STATE)


def make_pipe(clf, smote=False):
    steps = [("prep", make_preprocessor())]
    if smote:
        steps.append(("resample", make_sampler()))
    steps.append(("clf", clf))
    return ImbPipeline(steps)


def rf_cw():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1,
                                  class_weight="balanced_subsample", random_state=RANDOM_STATE)


def lgbm_cw():
    return LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          class_weight="balanced", random_state=RANDOM_STATE, verbose=-1, n_jobs=1)


def lr_cw():
    return LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)


# factories (a fresh, unfitted pipeline per fit)
MODELS = {
    # baselines
    "LogReg (class_weight)":       lambda: make_pipe(lr_cw()),
    "RandomForest (class_weight)": lambda: make_pipe(rf_cw()),
    "RandomForest + SMOTETomek":    lambda: make_pipe(RandomForestClassifier(
                                        n_estimators=300, min_samples_leaf=3, n_jobs=-1,
                                        random_state=RANDOM_STATE), smote=True),
    "LightGBM (class_weight)":     lambda: make_pipe(lgbm_cw()),
    # planned methods 
    "SPE (Self-Paced Ensemble)":   lambda: make_pipe(SelfPacedEnsembleClassifier(
                                        n_estimators=50, random_state=RANDOM_STATE)),
    "BalanceCascade":              lambda: make_pipe(BalanceCascadeClassifier(
                                        n_estimators=50, random_state=RANDOM_STATE)),
    # NOTE: imbens' SMOTEBagging uses plain SMOTE internally -> may interpolate ordinal codes
    "SMOTEBagging":                lambda: make_pipe(SMOTEBaggingClassifier(
                                        n_estimators=50, random_state=RANDOM_STATE)),
    "Stacking (class_weight)":     lambda: make_pipe(StackingClassifier(
                                        estimators=[("rf", rf_cw()), ("lgbm", lgbm_cw()), ("lr", lr_cw())],
                                        final_estimator=LogisticRegression(class_weight="balanced", max_iter=2000),
                                        cv=StratifiedKFold(10, shuffle=True, random_state=RANDOM_STATE),
                                        stack_method="predict_proba")),   # no external SMOTE (inner-CV leak)
}
if HAS_DEEP_FOREST:
    MODELS["Deep Forest + SMOTETomek"] = lambda: make_pipe(CascadeForestClassifier(
        max_layers=10, n_trees=100, n_jobs=-1, verbose=0, random_state=RANDOM_STATE), smote=True)
else:
    print("deep-forest not installed -> Deep Forest skipped (pip install deep-forest)")


def minority_report(y_true, y_pred, y_prob):
    return {
        "Precision (Class 1)": precision_score(y_true, y_pred, zero_division=0),
        "Recall (Class 1)":    recall_score(y_true, y_pred, zero_division=0),
        "F1 (Class 1)":        f1_score(y_true, y_pred, zero_division=0),
        "F2 (Class 1)":        fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "PR-AUC":              average_precision_score(y_true, y_prob),
        "ROC-AUC":             roc_auc_score(y_true, y_prob),
        "MCC":                 matthews_corrcoef(y_true, y_pred),
        "Balanced Accuracy":   balanced_accuracy_score(y_true, y_pred),
    }


def find_optimal_threshold(y_true, y_prob, beta=THRESH_BETA):
    scores = [fbeta_score(y_true, (y_prob >= t).astype(int), beta=beta, zero_division=0)
              for t in THRESH_GRID]
    i = int(np.argmax(scores))
    return float(THRESH_GRID[i]), float(scores[i])


def cv_oof(factory, X, y_np, seed_offset=0):
    oof = np.zeros((N_REPEATS, len(X)))
    fold_pr = []
    for r in range(N_REPEATS):
        skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE + seed_offset + r)
        for tr, va in skf.split(X, y_np):
            pipe = factory()
            pipe.fit(X.iloc[tr], y_np[tr])
            p = pipe.predict_proba(X.iloc[va])[:, 1]
            oof[r, va] = p
            fold_pr.append(average_precision_score(y_np[va], p))
    return oof, np.array(fold_pr)


def bootstrap_ci(y_true, y_prob, thr, n_boot=N_BOOT, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    rows = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        yt, yp = y_true[idx], y_prob[idx]
        rows.append(minority_report(yt, (yp >= thr).astype(int), yp))
    q = pd.DataFrame(rows).quantile([0.025, 0.975]).T
    q.columns = ["CI 2.5%", "CI 97.5%"]
    return q

MODELS.pop("Deep Forest + SMOTETomek", None)


# EXTRA-EXPERIMENT HELPERS
# Shared by E1-E6 (run at steps 4B, 4C, 7B, 7C, 8C, 8D). They reuse the SAME building blocks as the
# main pipeline (preprocessor, sampler, models, hyper-parameters, 10-fold x 3 CV) => comparable numbers.

X_ALL, Y_ALL = X.reset_index(drop=True), y.reset_index(drop=True)   # full de-duplicated data, ORIGINAL row order (E4 needs it)

# E2/E3/E4/E5 use the fast models only (Stacking is deliberately excluded: ~6 min per CV)
EXTRA_MODELS = {k: MODELS[k] for k in ["LogReg (class_weight)", "RandomForest (class_weight)",
                                       "LightGBM (class_weight)", "RandomForest + SMOTETomek"]}


def run_extra(tag, fn):
    if EXTRA_RUN == "none" or (EXTRA_RUN != "all" and tag not in EXTRA_RUN.split(",")):
        print(f"[extra] {tag} skipped (EXTRA_RUN={EXTRA_RUN})")
        return None
    t_x = time.perf_counter()
    print(f"\n[extra] ===== {tag.upper()} =====")
    try:
        out = fn()
        print(f"[extra] {tag.upper()} done in {(time.perf_counter() - t_x) / 60:.1f} min -> {EXTRA_DIR}/")
        return out
    except Exception as e:
        print(f"[extra] {tag.upper()} FAILED: {type(e).__name__}: {e}")
        return None


def make_preprocessor_subset(cols):
    sub_cat = [c for c in cat_cols if c in cols]
    sub_num = [c for c in num_cols if c in cols]
    parts = []
    if sub_cat:
        cats = [cat_categories[cat_cols.index(c)] for c in sub_cat]
        parts.append(("cat", SkPipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                         ("enc", OrdinalEncoder(categories=cats, handle_unknown="use_encoded_value",
                                                                unknown_value=-1))]), sub_cat))
    if sub_num:
        parts.append(("num", SkPipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), sub_num))
    return ColumnTransformer(parts, verbose_feature_names_out=False)


def cv_fold_scores(factory, X, y_np, n_splits=None, n_repeats=None, seed=0):
    n_splits = n_splits or N_SPLITS
    n_repeats = n_repeats or N_REPEATS
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE + seed)
    out = []
    for tr, va in rskf.split(X, y_np):
        pipe = factory()
        pipe.fit(X.iloc[tr], y_np[tr])
        p = pipe.predict_proba(X.iloc[va])[:, 1]
        out.append(average_precision_score(y_np[va], p))
    return np.array(out)

def bootstrap_ci_metric(y_true, y_prob, metric, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        vals.append(metric(y_true[idx], y_prob[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def nadeau_bengio_t(d, n_splits):
    d = np.asarray(d, float)
    k = len(d)
    var = d.var(ddof=1)
    if var == 0:
        return 0.0, 1.0
    n_test_over_train = 1.0 / (n_splits - 1)
    t = d.mean() / np.sqrt((1.0 / k + n_test_over_train) * var)
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return float(t), float(p)

def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj

deep-forest not installed -> Deep Forest skipped (pip install deep-forest)


In [ ]:
# CROSS-VALIDATION ON TRAIN ONLY -> OOF probs, thresholds, CV table
cv_rows, oof_store, thresholds = {}, {}, {}
for name, factory in MODELS.items():
    t0 = time.perf_counter()
    oof, fold_pr = cv_oof(factory, X_train, y_train_np)
    oof_mean = oof.mean(axis=0)                           # each sample's prediction always comes from a model that never saw it
    thr, oof_fb = find_optimal_threshold(y_train_np, oof_mean)
    pred = (oof_mean >= thr).astype(int)
    oof_store[name], thresholds[name] = oof_mean, thr
    cv_rows[name] = {
        "CV PR-AUC (mean)": fold_pr.mean(),               # <- model-selection criterion (threshold-free)
        "CV PR-AUC (std)":  fold_pr.std(),
        "OOF threshold":    thr,
        "OOF F1 @thr":      f1_score(y_train_np, pred, zero_division=0),
        "OOF Precision":    precision_score(y_train_np, pred, zero_division=0),
        "OOF Recall":       recall_score(y_train_np, pred, zero_division=0),
        "OOF MCC":          matthews_corrcoef(y_train_np, pred),
    }
    print(f"[CV] {name:<30} PR-AUC {fold_pr.mean():.4f}±{fold_pr.std():.4f} | "
          f"thr {thr:.2f} | OOF F1 {cv_rows[name]['OOF F1 @thr']:.4f}  ({time.perf_counter() - t0:.1f}s)")

df_cv = pd.DataFrame(cv_rows).T.sort_values("CV PR-AUC (mean)", ascending=False)
df_cv.to_csv(f"{OUT_DIR}/cv_summary.csv")
print("\n=== CV summary (train only) ===")
print(df_cv.round(4).to_string())
print(f"(random-classifier PR-AUC = {y_train.mean():.4f}; if models sit within ~1 std of each other, "
      f"the ranking is noise — prefer the simpler one)")


# EXTRA E2 — PAIRED STATISTICAL TESTS BETWEEN MODELS  (tests H3; p-values for the CV table)
# Same folds for every model => paired scores. Corrected resampled t-test (Nadeau & Bengio, 2003) + Holm correction.
def e2_paired_tests(reference="RandomForest (class_weight)"):
    X, y = X_ALL, Y_ALL.to_numpy()
    n_repeats = 1 if QUICK else N_REPEATS
    scores = {name: cv_fold_scores(f, X, y, n_repeats=n_repeats) for name, f in EXTRA_MODELS.items()}
    pd.DataFrame(scores).to_csv(f"{EXTRA_DIR}/e2_fold_scores.csv", index=False)

    rows = []
    others = [n for n in scores if n != reference]
    for n in others:
        d = scores[n] - scores[reference]
        t, p_nb = nadeau_bengio_t(d, N_SPLITS)
        try:
            p_w = stats.wilcoxon(d).pvalue
        except ValueError:                      # all diff = 0
            p_w = 1.0
        rows.append({"Model": n, "Reference": reference, "PR-AUC model": scores[n].mean(),
                     "PR-AUC ref": scores[reference].mean(), "Mean diff": d.mean(),
                     "t (Nadeau-Bengio)": t, "p (N-B)": p_nb, "p (Wilcoxon, NOT Corrected)": p_w})
    df = pd.DataFrame(rows)
    df["p (N-B) Holm"] = holm(df["p (N-B)"].to_numpy())
    df.to_csv(f"{EXTRA_DIR}/e2_paired_tests.csv", index=False)
    print("\n=== E2: Paired Test vs", reference, "===")
    print(df.round(4).to_string(index=False))
    print("Interpretation: p (N-B) Holm < 0.05 indicates a statistically significant difference"
          "Wilcoxon tests on repeated CV tend to be overly optimistic (overlapping folds); use them as supplementary evidence only.")
    return df

run_extra("e2", e2_paired_tests)


# EXTRA E1 — LEAKY vs LEAK-FREE PROTOCOL  (tests H1; Table III)
# How much does a leaky protocol inflate PR-AUC? A leak-free | B leaky scaling | C leaky global SMOTENC (C-all / C-orig)
# A2 in-fold SMOTENC without Tomek (clean comparator for C).
def e1_leaky_vs_leakfree():
    X, y = X_ALL, Y_ALL.to_numpy()
    n_repeats = 1 if QUICK else N_REPEATS
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=n_repeats, random_state=RANDOM_STATE)

    # A. leak-free
    res = {"A leak-free": [], "A2 in-fold SMOTENC (no Tomek)": [], "B leaky-scaling": [],
           "C-all leaky-SMOTE (all rows)": [], "C-orig leaky-SMOTE (original rows)": [],
           "A leak-free [LogReg]": [], "B leaky-scaling [LogReg]": []}
    for tr, va in rskf.split(X, y):
        pipe = make_pipe(rf_cw()); pipe.fit(X.iloc[tr], y[tr])
        res["A leak-free"].append(average_precision_score(y[va], pipe.predict_proba(X.iloc[va])[:, 1]))

    # A2. in-fold SMOTENC (no Tomek): same oversampler + SAME classifier as arm C, but resampling INSIDE the fold.
    #         => A2 vs C-orig isolates the leakage effect; A2 vs A isolates the oversampling effect.
    for tr, va in rskf.split(X, y):
        pipe = ImbPipeline([("prep", make_preprocessor()),
                            ("resample", SMOTENC(categorical_features=list(range(n_cat)), random_state=RANDOM_STATE)),
                            ("clf", RandomForestClassifier(n_estimators=100 if QUICK else 300, min_samples_leaf=3, n_jobs=-1,
                                                           class_weight="balanced_subsample", random_state=RANDOM_STATE))])
        pipe.fit(X.iloc[tr], y[tr])
        res["A2 in-fold SMOTENC (no Tomek)"].append(average_precision_score(y[va], pipe.predict_proba(X.iloc[va])[:, 1]))

    # B. leaky scaling (preprocessor on fit at all data)
    prep_global = make_preprocessor().fit(X)
    Xg = prep_global.transform(X)
    for tr, va in rskf.split(X, y):
        clf = rf_cw().fit(Xg[tr], y[tr])
        res["B leaky-scaling"].append(average_precision_score(y[va], clf.predict_proba(Xg[va])[:, 1]))

    # RF not sensitive to scale -> B Effect almost certainly ~0. LogReg (scale-sensitive) is used as a comparison.
    for tr, va in rskf.split(X, y):
        pipe = make_pipe(lr_cw()); pipe.fit(X.iloc[tr], y[tr])
        res["A leak-free [LogReg]"].append(average_precision_score(y[va], pipe.predict_proba(X.iloc[va])[:, 1]))
        clf = lr_cw().fit(Xg[tr], y[tr])
        res["B leaky-scaling [LogReg]"].append(average_precision_score(y[va], clf.predict_proba(Xg[va])[:, 1]))

    # C. Leaky SMOTENC on the entire dataset. Original rows = the first n rows (imblearn order).
    smote = SMOTENC(categorical_features=list(range(n_cat)), random_state=RANDOM_STATE)
    Xr, yr = smote.fit_resample(Xg, y)
    n_orig = len(Xg)
    is_orig = np.zeros(len(Xr), dtype=bool); is_orig[:n_orig] = True
    assert np.allclose(Xr[:n_orig], Xg), "assumption that the original row order = the first n rows does not hold"
    rskf_r = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=n_repeats, random_state=RANDOM_STATE)
    for tr, va in rskf_r.split(Xr, yr):
        clf = RandomForestClassifier(n_estimators=100 if QUICK else 300, min_samples_leaf=3, n_jobs=-1,
                                     class_weight="balanced_subsample", random_state=RANDOM_STATE)
        clf.fit(Xr[tr], yr[tr])
        p = clf.predict_proba(Xr[va])[:, 1]
        res["C-all leaky-SMOTE (all rows)"].append(average_precision_score(yr[va], p))
        m = is_orig[va]
        if yr[va][m].sum() > 0:
            res["C-orig leaky-SMOTE (original rows)"].append(average_precision_score(yr[va][m], p[m]))

    a2_mean = np.mean(res["A2 in-fold SMOTENC (no Tomek)"])
    rows = []
    for k, v in res.items():
        v = np.array(v)
        base = np.mean(res["A leak-free [LogReg]"] if "[LogReg]" in k else res["A leak-free"])
        rows.append({"Protokol": k, "PR-AUC mean": v.mean(), "PR-AUC std": v.std(), "n fold": len(v),
                     "Diff vs leak-free": v.mean() - base,
                     "Diff vs A2": (v.mean() - a2_mean) if k.startswith(("A2", "C-")) else np.nan})
    df = pd.DataFrame(rows)
    df.to_csv(f"{EXTRA_DIR}/e1_leaky_vs_leakfree.csv", index=False)
    print("\n=== E1: leaky vs leak-free (PR-AUC, RF class_weight) ===")
    print(df.round(4).to_string(index=False))
    print(f"(PR-AUC randomly on real data = {y.mean():.4f}; in the SMOTE results data = {yr.mean():.4f})")
    a_mean = np.mean(res["A leak-free"]); c_orig = np.mean(res["C-orig leaky-SMOTE (original rows)"])
    print(f"Effect oversampling only (A2 - A)         : {a2_mean - a_mean:+.4f}")
    print(f"Effect leak only   (C-orig - A2)     : {c_orig - a2_mean:+.4f}   <- clean number for H1")

    plt.figure(figsize=(9, 4.5))
    plt.barh(df["Protokol"], df["PR-AUC mean"], xerr=df["PR-AUC std"], color="#2b5c8f", edgecolor="black", lw=.5)
    plt.gca().invert_yaxis()
    plt.xlabel("PR-AUC (mean over folds, error bar = std)")
    plt.title("E1: scores inflate when the protocol leaks")
    plt.tight_layout(); plt.savefig(f"{EXTRA_DIR}/e1_leaky_vs_leakfree.png", dpi=200); plt.close()
    return df

run_extra("e1", e1_leaky_vs_leakfree)

[CV] LogReg (class_weight)          PR-AUC 0.2529±0.0726 | thr 0.67 | OOF F1 0.3258  (2.1s)
[CV] RandomForest (class_weight)    PR-AUC 0.2632±0.0845 | thr 0.39 | OOF F1 0.3223  (72.3s)
[CV] RandomForest + SMOTETomek      PR-AUC 0.2175±0.0743 | thr 0.36 | OOF F1 0.2520  (103.6s)
[CV] LightGBM (class_weight)        PR-AUC 0.2559±0.0865 | thr 0.34 | OOF F1 0.2902  (20.3s)
[CV] SPE (Self-Paced Ensemble)      PR-AUC 0.2389±0.0721 | thr 0.59 | OOF F1 0.3313  (23.5s)
[CV] BalanceCascade                 PR-AUC 0.2000±0.0757 | thr 0.55 | OOF F1 0.2740  (18.5s)
[CV] SMOTEBagging                   PR-AUC 0.2425±0.0859 | thr 0.30 | OOF F1 0.2918  (48.5s)
[CV] Stacking (class_weight)        PR-AUC 0.2729±0.0851 | thr 0.75 | OOF F1 0.3238  (733.3s)

=== CV summary (train only) ===
                             CV PR-AUC (mean)  CV PR-AUC (std)  OOF threshold  OOF F1 @thr  OOF Precision  OOF Recall  OOF MCC
Stacking (class_weight)                0.2729           0.0851           0.75       0.3238     

,Protokol,PR-AUC mean,PR-AUC std,n fold,Diff vs leak-free,Diff vs A2
0,A leak-free,0.245302,0.068148,30,0.000000,NaN
1,A2 in-fold SMOTENC (no Tomek),0.205269,0.058952,30,-0.040033,0.000000
2,B leaky-scaling,0.245000,0.067988,30,-0.000303,NaN
3,C-all leaky-SMOTE (all rows),0.975411,0.005655,30,0.730108,0.770142
4,C-orig leaky-SMOTE (original rows),0.558359,0.094748,30,0.313057,0.353090
5,A leak-free [LogReg],0.235280,0.067645,30,0.000000,NaN
6,B leaky-scaling [LogReg],0.235453,0.068084,30,0.000173,NaN


In [ ]:
# MODEL SELECTION — from CV only (all models untuned = fair comparison)
best_name = df_cv.index[0]
print(f"\n>>> Selected (CV PR-AUC, default hyperparameters): {best_name}")

# HYPERPARAMETER TUNING — CV-selected model ONLY, on TRAIN ONLY
# Search spaces use sklearn's  <step>__<sub-estimator>__<param>  naming (step "clf" = the classifier).
SEARCH_SPACES = {
    "Stacking (class_weight)": {
        "clf__rf__n_estimators":        randint(200, 601),
        "clf__rf__max_depth":           [None, 4, 6, 8, 12],
        "clf__rf__min_samples_leaf":    randint(1, 11),
        "clf__rf__max_features":        ["sqrt", 0.5],
        "clf__lgbm__n_estimators":      randint(100, 501),
        "clf__lgbm__learning_rate":     loguniform(0.01, 0.1),
        "clf__lgbm__num_leaves":        [7, 15, 31],
        "clf__lgbm__min_child_samples": randint(5, 41),
        "clf__lgbm__subsample":         uniform(0.6, 0.4),
        "clf__lgbm__colsample_bytree":  uniform(0.6, 0.4),
        "clf__lgbm__reg_lambda":        loguniform(1e-2, 10),
        "clf__lr__C":                   loguniform(1e-2, 10),
        "clf__final_estimator__C":      loguniform(1e-2, 10),
        "clf__passthrough":             [False, True],
    },
    "LightGBM (class_weight)": {
        "clf__n_estimators": randint(100, 501), "clf__learning_rate": loguniform(0.01, 0.1),
        "clf__num_leaves": [7, 15, 31], "clf__min_child_samples": randint(5, 41),
        "clf__subsample": uniform(0.6, 0.4), "clf__colsample_bytree": uniform(0.6, 0.4),
        "clf__reg_lambda": loguniform(1e-2, 10),
    },
    "RandomForest (class_weight)": {
        "clf__n_estimators": randint(200, 601), "clf__max_depth": [None, 4, 6, 8, 12],
        "clf__min_samples_leaf": randint(1, 11), "clf__max_features": ["sqrt", 0.5],
    },
}


def make_tuned_factory(name, params):
    def factory():
        pipe = MODELS[name]()
        pipe.set_params(**params)
        return pipe
    return factory


FINAL_FACTORIES = dict(MODELS)                            # what actually gets fit on full train
tune_name = best_name if TUNE_MODEL == "auto" else (None if TUNE_MODEL == "none" else TUNE_MODEL)
tuning_info = {"tuned": False}
if tune_name is not None and tune_name not in MODELS:
    raise ValueError(f"TUNE_MODEL={tune_name!r} is not a model name. Options: {list(MODELS)}")
if tune_name is not None and tune_name != best_name:      # only possible via an explicit TUNE_MODEL override
    print(f"NOTE: explicit TUNE_MODEL override -> final model forced from '{best_name}' to '{tune_name}'")
    best_name = tune_name

if tune_name in SEARCH_SPACES:
    print(f"\n=== Tuning {tune_name}: {TUNE_N_ITER} candidates x {N_SPLITS}-fold x {TUNE_REPEATS} repeats ===")
    t_tune = time.perf_counter()
    search = RandomizedSearchCV(
        estimator=MODELS[tune_name](),                    # full pipeline => encoder/scaler/resampler refit per inner fold
        param_distributions=SEARCH_SPACES[tune_name],
        n_iter=TUNE_N_ITER,
        scoring="average_precision",                      # same threshold-free criterion as model selection
        cv=RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=TUNE_REPEATS, random_state=RANDOM_STATE + 50),
        refit=False,                                      # we refit ourselves on full train below
        n_jobs=TUNE_N_JOBS, random_state=RANDOM_STATE, error_score="raise",
    )
    search.fit(X_train, y_train_np)                       # TRAIN ONLY — test set never enters
    pd.DataFrame(search.cv_results_).sort_values("rank_test_score").to_csv(f"{OUT_DIR}/tuning_cv_results.csv", index=False)
    print(f"Search done in {(time.perf_counter() - t_tune) / 60:.1f} min | best inner-CV PR-AUC "
          f"{search.best_score_:.4f} (optimistic: it was picked as the max of {TUNE_N_ITER} noisy scores)")

    # Re-check on FRESH CV splits (different seeds than the search and than the first CV).
    # The tuned config was selected on the search splits, so this comparison is less biased.
    tuned_factory = make_tuned_factory(tune_name, search.best_params_)
    oof_def, pr_def = cv_oof(MODELS[tune_name], X_train, y_train_np, seed_offset=100)
    oof_tun, pr_tun = cv_oof(tuned_factory, X_train, y_train_np, seed_offset=100)
    print(f"Fresh-split CV PR-AUC | default: {pr_def.mean():.4f}±{pr_def.std():.4f} | "
          f"tuned: {pr_tun.mean():.4f}±{pr_tun.std():.4f}")

    adopt = pr_tun.mean() > pr_def.mean()
    FINAL_FACTORIES[tune_name] = tuned_factory if adopt else MODELS[tune_name]
    oof_final = (oof_tun if adopt else oof_def).mean(axis=0)
    thresholds[tune_name], _ = find_optimal_threshold(y_train_np, oof_final)   # threshold from fresh OOF of the ADOPTED config
    tuning_info = {
        "tuned": bool(adopt), "model": tune_name, "best_params": {k: (v.item() if hasattr(v, "item") else v)
                                                                   for k, v in search.best_params_.items()},
        "search_best_inner_cv_pr_auc": float(search.best_score_),
        "fresh_cv_pr_auc_default": float(pr_def.mean()), "fresh_cv_pr_auc_tuned": float(pr_tun.mean()),
    }
    json.dump(tuning_info, open(f"{OUT_DIR}/tuning_summary.json", "w"), indent=2, default=str)
    print(f">>> {'ADOPTED tuned params' if adopt else 'Tuned params did NOT beat default on fresh splits -> keeping default'}")
    if adopt:
        print(f"    {tuning_info['best_params']}")
else:
    print(f"\nNo tuning (TUNE_MODEL={TUNE_MODEL!r}, target={tune_name!r} has no search space).")

best_thr = thresholds[best_name]
print(f">>> Final config: {best_name} | OOF threshold = {best_thr:.2f}")

# FINAL FIT ON FULL TRAIN -> ONE-SHOT TEST EVALUATION
def fit_final(name):
    pipe = FINAL_FACTORIES[name]()
    pipe.fit(X_train, y_train_np)
    return pipe


def evaluate_test(pipe, thr):
    prob = pipe.predict_proba(X_test)[:, 1]
    pred = (prob >= thr).astype(int)
    return prob, pred, minority_report(y_test_np, pred, prob), confusion_matrix(y_test_np, pred)


final_pipe = fit_final(best_name)
prob_te, pred_te, rep, cm = evaluate_test(final_pipe, best_thr)
ci = bootstrap_ci(y_test_np, prob_te, best_thr)

df_test = pd.DataFrame({"Test score": pd.Series(rep)}).join(ci)
df_test.to_csv(f"{OUT_DIR}/test_results_best.csv")
print(f"\n=== TEST SET (evaluated once) — {best_name}, threshold {best_thr:.2f} ===")
print(df_test.round(4).to_string())
print(f"Confusion matrix (rows=true, cols=pred) [TN FP / FN TP]:\n{cm}")

# single-sample inference latency of the full deployed pipeline (preprocessing + model)
lat = []
for i in range(min(100, len(X_test))):
    t = time.perf_counter()
    final_pipe.predict_proba(X_test.iloc[[i]])
    lat.append((time.perf_counter() - t) * 1000)
print(f"Latency (full pipeline, 1 sample): {np.mean(lat):.2f} ms (±{np.std(lat):.2f})")

if REPORT_ALL_ON_TEST:                                    # descriptive only — NEVER use to re-select
    rows = {}
    for name in MODELS:
        p = final_pipe if name == best_name else fit_final(name)
        rows[name] = evaluate_test(p, thresholds[name])[2]
    pd.DataFrame(rows).T.to_csv(f"{OUT_DIR}/test_results_all_DESCRIPTIVE.csv")
    print("All-model test table saved (descriptive; selection was already done on CV).")

# plots: PR curve + confusion matrix
prec_c, rec_c, _ = precision_recall_curve(y_test_np, prob_te)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(rec_c, prec_c, color="#2b5c8f", lw=2)
ax[0].axhline(y_test.mean(), ls="--", color="grey", label=f"random ({y_test.mean():.3f})")
ax[0].set(xlabel="Recall", ylabel="Precision", title=f"PR curve — {best_name}\nAUPRC={rep['PR-AUC']:.3f}")
ax[0].legend()
ax[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[1].text(j, i, int(v), ha="center", va="center", fontsize=14)
ax[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Non-Haz", "Haz"], yticklabels=["Non-Haz", "Haz"],
          xlabel="Predicted", ylabel="True", title=f"Confusion matrix @ t={best_thr:.2f}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/test_pr_curve_confusion.png", dpi=300)
plt.close()


# EXTRA E4 — TEMPORAL SPLIT vs RANDOM SPLIT  (limitation check)
# Only meaningful if the CSV rows are still in original time order. Threshold from OOF on the train part only.
def e4_temporal_split(model_names=("RandomForest (class_weight)", "LogReg (class_weight)")):
    X, y = X_ALL, Y_ALL.to_numpy()
    cut = int(len(X) * (1 - TEST_SIZE))
    Xtr, Xte, ytr, yte = X.iloc[:cut].reset_index(drop=True), X.iloc[cut:].reset_index(drop=True), y[:cut], y[cut:]
    Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    Xtr_s, Xte_s = Xtr_s.reset_index(drop=True), Xte_s.reset_index(drop=True)
    print(f"\nTemporal: train={len(Xtr)} (pos={ytr.sum()}), test={len(Xte)} (pos={yte.sum()})")
    print(f"Acak    : train={len(Xtr_s)} (pos={ytr_s.sum()}), test={len(Xte_s)} (pos={yte_s.sum()})")
    if yte.sum() < 10:
        print("WARNING: If the temporal test is positive and the value is < 10, the confidence interval will be very wide.")

    grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
    rows = []
    for name in model_names:
        for tag, (A, ya, B, yb) in {"temporal": (Xtr, ytr, Xte, yte), "random (stratified)": (Xtr_s, ytr_s, Xte_s, yte_s)}.items():
            skf = StratifiedKFold(5 if QUICK else 10, shuffle=True, random_state=RANDOM_STATE)
            oof = np.zeros(len(A))
            for tr, va in skf.split(A, ya):
                p = EXTRA_MODELS[name](); p.fit(A.iloc[tr], ya[tr]); oof[va] = p.predict_proba(A.iloc[va])[:, 1]
            thr = float(grid[np.argmax([f1_score(ya, (oof >= t).astype(int), zero_division=0) for t in grid])])
            fin = EXTRA_MODELS[name](); fin.fit(A, ya)
            prob = fin.predict_proba(B)[:, 1]
            pred = (prob >= thr).astype(int)
            lo, hi = bootstrap_ci_metric(yb, prob, average_precision_score, n_boot=200 if QUICK else 1000)
            rows.append({"Model": name, "Split": tag, "n pos test": int(yb.sum()), "threshold OOF": thr,
                         "PR-AUC": average_precision_score(yb, prob), "CI 2.5%": lo, "CI 97.5%": hi,
                         "ROC-AUC": roc_auc_score(yb, prob), "Precision": precision_score(yb, pred, zero_division=0),
                         "Recall": recall_score(yb, pred, zero_division=0), "F1": f1_score(yb, pred, zero_division=0),
                         "PR-AUC acak (prevalensi)": yb.mean()})
    df = pd.DataFrame(rows)
    df.to_csv(f"{EXTRA_DIR}/e4_temporal_vs_random.csv", index=False)
    print("\n=== E4: temporal vs random ===")
    print(df.round(4).to_string(index=False))
    return df

run_extra("e4", e4_temporal_split)


# EXTRA E5 — COST ANALYSIS OF THE THRESHOLD  (operational implication)
# Cost ratios are ASSUMPTIONS (sensitivity analysis), not real mine costs.
def e5_cost_analysis(model_name="RandomForest (class_weight)", cost_ratios=(1, 2, 5, 10, 20, 50)):
    X, y = X_ALL, Y_ALL.to_numpy()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    skf = StratifiedKFold(5 if QUICK else 10, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(Xtr))
    for tr, va in skf.split(Xtr, ytr):
        p = EXTRA_MODELS[model_name](); p.fit(Xtr.iloc[tr], ytr[tr]); oof[va] = p.predict_proba(Xtr.iloc[va])[:, 1]
    fin = EXTRA_MODELS[model_name](); fin.fit(Xtr, ytr)
    prob = fin.predict_proba(Xte)[:, 1]

    grid = np.round(np.arange(0.01, 1.0, 0.01), 2)

    def cost(yt, yp, thr, r):
        pred = (yp >= thr).astype(int)
        fp = int(((pred == 1) & (yt == 0)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
        return fp * 1.0 + fn * r, fp, fn

    # bootstrap for the saving: stratified resampling of the TEST rows (n pos / n neg fixed), threshold FIXED from OOF
    n_boot = 200 if QUICK else 1000
    rng_b = np.random.default_rng(RANDOM_STATE)
    pos_i, neg_i = np.where(yte == 1)[0], np.where(yte == 0)[0]
    boot_idx = [np.concatenate([rng_b.choice(pos_i, len(pos_i)), rng_b.choice(neg_i, len(neg_i))]) for _ in range(n_boot)]

    rows = []
    for r in cost_ratios:
        best_thr = float(grid[np.argmin([cost(ytr, oof, t, r)[0] for t in grid])])
        c_te, fp, fn = cost(yte, prob, best_thr, r)
        c_never = yte.sum() * r                       # never trigger an alarm: all bumps = FN
        c_always = (yte == 0).sum() * 1.0             # Always trigger an alarm: all non-bumps = FP
        pred = (prob >= best_thr).astype(int)
        saves = [min(c_never, c_always) - cost(yte[idx], prob[idx], best_thr, r)[0] for idx in boot_idx]
        save_lo, save_hi = np.percentile(saves, [2.5, 97.5])
        rows.append({"Rasio FN:FP": r, "threshold (OOF)": best_thr, "FP test": fp, "FN test": fn,
                     "Recall test": recall_score(yte, pred, zero_division=0),
                     "Precision test": precision_score(yte, pred, zero_division=0),
                     "Model cost": c_te, "Cost 'never alarm'": c_never, "Cost 'always alarm'": c_always,
                     "Savings vs best baseline": min(c_never, c_always) - c_te,
                     "2.5% CI savings": save_lo, "97.5% CI savings": save_hi})
    df = pd.DataFrame(rows)
    df.to_csv(f"{EXTRA_DIR}/e5_cost_analysis.csv", index=False)
    print(f"\n=== E5: Cost analysis ({model_name}; positif test = {int(yte.sum())}) ===")
    print(df.round(3).to_string(index=False))
    print("Interpretation: 'Savings vs. best baseline' > 0 means the model is less costly than the "
          "simple baseline policy. If <= 0 at a given ratio, the model provides no operational value at that ratio.")
    print("Savings CI = stratified bootstrap on test rows (threshold fixed from OOF); it reflects only test-sample "
          f"noise ({int(yte.sum())} positif cases), not uncertainty in threshold selection.")

    plt.figure(figsize=(7, 4))
    plt.plot(df["Ratio FN:FP"], df["Model cost"], "o-", label="model (OOF-tuned threshold)")
    plt.plot(df["Ratio FN:FP"], df["Cost 'never alarm'"], "s--", label="never alarm")
    plt.plot(df["Ratio FN:FP"], df["Cost 'always alarm'"], "^--", label="always alarm")
    plt.xscale("log"); plt.xlabel("Cost ratio FN : FP"); plt.ylabel("Total relative cost (test set)")
    plt.title("E5: operational value of the model"); plt.legend(); plt.tight_layout()
    plt.savefig(f"{EXTRA_DIR}/e5_cost_analysis.png", dpi=200); plt.close()
    return df

run_extra("e5", e5_cost_analysis)


>>> Selected (CV PR-AUC, default hyperparameters): Stacking (class_weight)

=== Tuning Stacking (class_weight): 25 candidates x 10-fold x 2 repeats ===
Search done in 202.5 min | best inner-CV PR-AUC 0.2904 (optimistic: it was picked as the max of 25 noisy scores)
Fresh-split CV PR-AUC | default: 0.2851±0.0880 | tuned: 0.2856±0.0845
>>> ADOPTED tuned params
    {'clf__final_estimator__C': 0.1911275821777789, 'clf__lgbm__colsample_bytree': 0.9272059063689972, 'clf__lgbm__learning_rate': 0.07256556536786182, 'clf__lgbm__min_child_samples': 10, 'clf__lgbm__n_estimators': 236, 'clf__lgbm__num_leaves': 15, 'clf__lgbm__reg_lambda': 0.04637921903458034, 'clf__lgbm__subsample': 0.6479461469334731, 'clf__lr__C': 0.10300196600986772, 'clf__passthrough': False, 'clf__rf__max_depth': 6, 'clf__rf__max_features': 0.5, 'clf__rf__min_samples_leaf': 8, 'clf__rf__n_estimators': 432}
>>> Final config: Stacking (class_weight) | OOF threshold = 0.71

=== TEST SET (evaluated once) — Stacking (class_weight)

In [ ]:
# XAI — post-hoc only, results never fed back into training/selection
# Permutation importance on RAW features, through the entire fitted pipeline
pi = permutation_importance(final_pipe, X_test, y_test_np, scoring="average_precision",
                            n_repeats=30, random_state=RANDOM_STATE, n_jobs=1)
df_pi = pd.DataFrame({"mean": pi.importances_mean, "std": pi.importances_std},
                     index=X_test.columns).sort_values("mean")
df_pi.to_csv(f"{OUT_DIR}/permutation_importance.csv")
plt.figure(figsize=(9, 6))
plt.barh(df_pi.index, df_pi["mean"], xerr=df_pi["std"], color="#2b5c8f", edgecolor="black", lw=0.5)
plt.xlabel("Drop in PR-AUC when feature is permuted (test set, 30 repeats)")
plt.title(f"Permutation importance — {best_name}\n(noisy: only {int(y_test.sum())} positives in test)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/permutation_importance.png", dpi=300)
plt.close()

# SHAP — TreeExplainer needs a single tree model. Use the best CV-ranked tree model.
SHAP_OK = ["RandomForest (class_weight)", "RandomForest + SMOTETomek", "LightGBM (class_weight)"]
if HAS_SHAP:
    try:
        explain_name = next(n for n in df_cv.index if n in SHAP_OK)
        explain_pipe = final_pipe if explain_name == best_name else fit_final(explain_name)
        prep, clf = explain_pipe.named_steps["prep"], explain_pipe.named_steps["clf"]
        Xt = pd.DataFrame(prep.transform(X_test), columns=prep.get_feature_names_out(), index=X_test.index)
        sv = shap.TreeExplainer(clf).shap_values(Xt)
        if isinstance(sv, list):
            sv = sv[1]
        elif getattr(sv, "ndim", 2) == 3:
            sv = sv[:, :, 1]
        shap.summary_plot(sv, Xt, show=False)
        plt.title(f"SHAP — {explain_name} (class 1 = Hazardous)")
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/shap_summary.png", dpi=300, bbox_inches="tight")
        plt.close()
        print(f"SHAP saved (explained model: {explain_name})")
    except Exception as e:
        print(f"SHAP skipped: {e}")


# EXTRA E3 — FEATURE-IMPORTANCE CONSISTENCY ACROSS FOLDS AND MODELS  (tests H2)
# Permutation importance on EVERY validation fold (5-fold x 4 repeats), not on one small test set.
def e3_feature_consistency(model_names=("RandomForest (class_weight)", "LightGBM (class_weight)")):
    X, y = X_ALL, Y_ALL.to_numpy()
    repeats = 1 if QUICK else 4
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=repeats, random_state=RANDOM_STATE)
    all_tables = {}
    for name in model_names:
        imps = []
        for tr, va in rskf.split(X, y):
            pipe = EXTRA_MODELS[name](); pipe.fit(X.iloc[tr], y[tr])
            pi = permutation_importance(pipe, X.iloc[va], y[va], scoring="average_precision",
                                        n_repeats=5 if QUICK else 10, random_state=RANDOM_STATE, n_jobs=1)
            imps.append(pi.importances_mean)
        M = pd.DataFrame(imps, columns=X.columns)               # row = fold, column = fitur
        ranks = M.rank(axis=1, ascending=False)
        tab = pd.DataFrame({"importance mean": M.mean(), "importance std": M.std(),
                            "rank mean": ranks.mean(), "top3 freq": (ranks <= 3).mean(),
                            "% fold importance > 0": (M > 0).mean()}).sort_values("rank mean")
        tab.to_csv(f"{EXTRA_DIR}/e3_importance_{name.split()[0]}.csv")
        all_tables[name] = tab
        print(f"\n=== E3: consistency feature - {name} ({len(M)} fold) ===")
        print(tab.round(4).head(10).to_string())

        top = tab.head(10).iloc[::-1]
        plt.figure(figsize=(8, 5))
        plt.barh(top.index, top["importance mean"], xerr=top["importance std"], color="#2b5c8f",
                 edgecolor="black", lw=.5)
        plt.xlabel("Drop in PR-AUC when the feature is permuted (mean over folds)")
        plt.title(f"E3: permutation importance - {name}")
        plt.tight_layout(); plt.savefig(f"{EXTRA_DIR}/e3_importance_{name.split()[0]}.png", dpi=200); plt.close()

    # Inter-model rank agreement (Spearman)
    if len(all_tables) >= 2:
        a, b = list(all_tables.values())[:2]
        rho, p = stats.spearmanr(a["mean rank"], b.loc[a.index, "mean rank"])
        print(f"\nFeature rank agreement between 2 models: Spearman rho = {rho:.3f} (p = {p:.4f})")
    return all_tables

run_extra("e3", e3_feature_consistency)


# EXTRA E6 — FEATURE-GROUP ABLATION: PULSE vs ENERGY vs BUMPS vs QUALITATIVE  (tests H2; Table IV)
# Same model, same folds, different feature subsets; paired test against the all-features model (Nadeau-Bengio + Holm).
FEATURE_GROUPS = {
    "pulse":       ["gpuls", "gdpuls"],
    "energy":      ["genergy", "gdenergy", "energy", "maxenergy"],
    "bumps":       ["nbumps", "nbumps2", "nbumps3", "nbumps4", "nbumps5"],
    "qualitative": ["seismic", "seismoacoustic", "hazard", "shift"],
}


def e6_group_ablation(base="RandomForest (class_weight)"):
    X, y = X_ALL, Y_ALL.to_numpy()
    n_repeats = 1 if QUICK else N_REPEATS
    groups = {g: [c for c in cols if c in X.columns] for g, cols in FEATURE_GROUPS.items()}
    rest = [c for c in X.columns if c not in sum(groups.values(), [])]
    if rest:
        print("Features that do not belong to any group (remain in “all”)'):", rest)

    def factory_for(cols):
        base_clf = {"RandomForest (class_weight)": rf_cw, "LightGBM (class_weight)": lgbm_cw,
                    "LogReg (class_weight)": lr_cw}[base]
        return lambda: ImbPipeline([("prep", make_preprocessor_subset(cols)), ("clf", base_clf())])

    configs = {"all feature": list(X.columns)}
    for g, cols in groups.items():
        configs[f"only {g}"] = cols
    for g, cols in groups.items():
        configs[f"all execpt {g}"] = [c for c in X.columns if c not in cols]

    scores = {k: cv_fold_scores(factory_for(v), X, y, n_repeats=n_repeats) for k, v in configs.items()}
    pd.DataFrame(scores).to_csv(f"{EXTRA_DIR}/e6_fold_scores.csv", index=False)
    ref = scores["all feature"]
    rows = []
    for k, v in scores.items():
        if k == "all feature":
            rows.append({"configuration": k, "n feature": len(configs[k]), "PR-AUC mean": v.mean(), "PR-AUC std": v.std(),
                         "Diff vs all": 0.0, "p (N-B)": np.nan})
            continue
        d = v - ref
        _, p = nadeau_bengio_t(d, N_SPLITS)
        rows.append({"Configuration": k, "n fitur": len(configs[k]), "PR-AUC mean": v.mean(), "PR-AUC std": v.std(),
                     "Diff vs all": d.mean(), "p (N-B)": p})
    df = pd.DataFrame(rows)
    mask = df["p (N-B)"].notna()
    df.loc[mask, "p (N-B) Holm"] = holm(df.loc[mask, "p (N-B)"].to_numpy())
    df.to_csv(f"{EXTRA_DIR}/e6_group_ablation.csv", index=False)
    print(f"\n=== E6: Feature Cluster Ablation ({base}) ===")
    print(df.round(4).to_string(index=False))
    print(f"(Random PR-AUC = {y.mean():.4f}). Only claim that “Group X is significant” if p-value < 0.05 AND "
          "The difference is large enough to be practically significant.")
    return df

run_extra("e6", e6_group_ablation)

SHAP saved (explained model: RandomForest (class_weight))

[extra] ===== E3 =====

=== E3: consistency feature - RandomForest (class_weight) (20 fold) ===
           importance mean  importance std  rank mean  top3 freq  % fold importance > 0
gpuls               0.0557          0.0344       1.50       0.95                   1.00
genergy             0.0286          0.0278       3.85       0.65                   0.85
nbumps              0.0109          0.0196       5.95       0.30                   0.80
nbumps2             0.0111          0.0174       6.45       0.30                   0.85
energy              0.0147          0.0241       6.55       0.30                   0.75
gdpuls              0.0054          0.0142       7.85       0.10                   0.65
nbumps3             0.0046          0.0125       8.35       0.20                   0.60
seismic             0.0036          0.0146       9.10       0.15                   0.65
shift               0.0019          0.0080       9.25

,configuration,n feature,PR-AUC mean,PR-AUC std,Diff vs all,p (N-B),Configuration,n fitur,p (N-B) Holm
0,all feature,15.0,0.245302,0.068148,0.000000,NaN,NaN,NaN,NaN
1,NaN,NaN,0.150479,0.039904,-0.094823,0.000089,only pulse,2.0,0.000537
2,NaN,NaN,0.182640,0.049738,-0.062663,0.016400,only energy,4.0,0.065598
3,NaN,NaN,0.149588,0.038812,-0.095715,0.000076,only bumps,5.0,0.000535
4,NaN,NaN,0.102416,0.017634,-0.142886,0.000003,only qualitative,4.0,0.000028
5,NaN,NaN,0.206612,0.051199,-0.038690,0.010851,all execpt pulse,13.0,0.054257
6,NaN,NaN,0.230667,0.067167,-0.014636,0.336490,all execpt energy,11.0,0.672979
7,NaN,NaN,0.243010,0.074082,-0.002292,0.881510,all execpt bumps,10.0,0.881510
8,NaN,NaN,0.229797,0.057680,-0.015505,0.170690,all execpt qualitative,11.0,0.512069


In [ ]:
# SAVE THE WHOLE ARTIFACT (preprocessing + model + threshold)
artifact = {
    "pipeline": final_pipe, "threshold": best_thr, "model_name": best_name,
    "feature_columns": X_train.columns.tolist(), "tuning": tuning_info,
    "versions": {"sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
path = f"{OUT_DIR}/best_model_artifact.pkl"
joblib.dump(artifact, path)

loaded = joblib.load(path)                                # round-trip sanity check on RAW input
chk = (loaded["pipeline"].predict_proba(X_test[loaded["feature_columns"]])[:, 1] >= loaded["threshold"]).astype(int)
assert np.array_equal(chk, pred_te), "Reloaded artifact disagrees with in-memory pipeline!"
print(f"Artifact saved + verified -> {path}")
print(f"\nDONE in {(time.perf_counter() - t0_all) / 60:.1f} min")

Artifact saved + verified -> results/best_model_artifact.pkl

DONE in 424.1 min
